# OAuth Permission Abuse & Service Principal Escalation Graph

> **Platform:** Microsoft Sentinel Custom Graph | **Domain:** Identity Security
> **Data Sources:** EntraServicePrincipals, AuditLogs, AADServicePrincipalSignInLogs, AADRiskyServicePrincipals, AADRiskyUsers, AADServicePrincipalRiskEvents, MicrosoftGraphActivityLogs
> **Nodes:** 9 (ServicePrincipal, AppPermission, User, Resource, SourceIP, DirectoryRole, RiskEvent, EntraGroup, SensitiveEndpoint)
> **Edges:** 11 (HasPermission, GrantedBy, CalledAPI, AuthenticatedAs, GrantedPermissionTo, HasDirectoryRole, OwnsApp, AddedSecretTo, HasRiskEvent, MemberOf, CalledSensitiveAPI)

Traces OAuth consent chains, service principal credential abuse, and transitive privilege escalation paths. Identifies consent hub users, over-permissioned service principals, and credential backdoor patterns that span the identity-application-resource boundary.

## Environment & Configuration

Import required packages, configure the Sentinel Data Lake provider, and set graph parameters.


In [ ]:
# =============================================================================
# Cell 2: Imports & Configuration
# =============================================================================

from sentinel_lake.providers import MicrosoftSentinelProvider
from sentinel_graph import GraphSpecBuilder, Graph
from pyspark.sql.functions import (
 col, lit, expr, coalesce, concat_ws, count, sum as spark_sum,
 explode, from_json, get_json_object, regexp_extract,
 to_date, datediff, current_timestamp, collect_set, first,
 when, isnan, isnull, lower, split, size, array_contains,
 min as spark_min, max as spark_max
)
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, MapType
from pyspark.sql.window import Window

# ⚠️ REQUIRED: Set your Microsoft Sentinel workspace name before running this notebook
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>" # ← Replace with your Sentinel workspace name
# 📅 OPTIONAL: Adjust the lookback window (default: 7 days). Use 1 for IR, 14 for hunting, 30+ for historical.
TIME_WINDOW_DAYS = 1

# Sensitive API patterns for CalledSensitiveAPI edge (P2.4)
SENSITIVE_API_PATTERNS = [
 "/roleManagement/",
 "/directoryRoles/",
 "/appRoleAssignments",
 "/oauth2PermissionGrants",
 "/servicePrincipals/*/addPassword",
 "/servicePrincipals/*/addKey",
 "/applications/*/addPassword",
 "/applications/*/addKey",
 "/applications/*/owners",
 "/servicePrincipals/*/owners",
 "/privilegedAccess/",
 "/policies/"
]

# Tier Zero directory roles (P0.3)
TIER_ZERO_ROLES = [
 "global administrator",
 "privileged role administrator",
 "privileged authentication administrator",
 "application administrator",
 "cloud application administrator",
 "partner tier2 support",
 "security administrator"
]

# --- Initialize providers ---
sentinel_provider = MicrosoftSentinelProvider(spark)

print(f"⚙️ Workspace: {WORKSPACE_NAME}")
print(f"⚙️ Time window: {TIME_WINDOW_DAYS} days")


## Data Ingestion

Read source tables from the Sentinel Data Lake, apply time filters, and persist to Spark memory.


In [ ]:
# =============================================================================
# Cell 3: Read EntraServicePrincipals → Build ServicePrincipal Nodes
# [P1.3] Enrich with SP Risk Level from AADRiskyServicePrincipals
# [P2.2] Enrich with credential type + sign-in metadata
# =============================================================================

# Read service principal inventory
_sp_raw = sentinel_provider.read_table("EntraServicePrincipals")
_sp_raw = _sp_raw.df if hasattr(_sp_raw, 'df') else _sp_raw

sp_raw_df = (
 _sp_raw
 .filter(col("TimeGenerated") >= expr("current_timestamp() - INTERVAL 1 DAYS"))
 .filter((col("id").isNotNull()) & (col("id") != ""))
 .select(
 "id", "appId", "displayName", "servicePrincipalType",
 "accountEnabled", "appOwnerOrganizationId", "tenantId"
 )
 .dropDuplicates(["id"])
).persist()

print(f"📊 ServicePrincipals loaded: {sp_raw_df.count()}")

# --- [P1.3] Read AADRiskyServicePrincipals for risk enrichment ---
_risky_sp_raw = sentinel_provider.read_table("AADRiskyServicePrincipals", WORKSPACE_NAME)
_risky_sp_raw = _risky_sp_raw.df if hasattr(_risky_sp_raw, 'df') else _risky_sp_raw

risky_sp_df = (
 _risky_sp_raw
 .filter(col("TimeGenerated") >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
 .select(
 col("Id").alias("RiskSpId"),
 col("RiskLevel").alias("SpRiskLevel"),
 col("RiskState").alias("SpRiskState"),
 col("RiskDetail").alias("SpRiskDetail")
 )
 .dropDuplicates(["RiskSpId"])
)
print(f"📊 Risky SPs loaded: {risky_sp_df.count()}")

# --- [P2.2] Read AADServicePrincipalSignInLogs for credential metadata ---
_sp_signin_meta = sentinel_provider.read_table("AADServicePrincipalSignInLogs", WORKSPACE_NAME)
_sp_signin_meta = _sp_signin_meta.df if hasattr(_sp_signin_meta, 'df') else _sp_signin_meta

sp_cred_metadata_df = (
 _sp_signin_meta
 .filter(col("TimeGenerated") >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
 .groupBy("ServicePrincipalId")
 .agg(
 first(col("ClientCredentialType")).alias("PrimaryCredentialType"),
 spark_max(col("TimeGenerated")).alias("LastSignInDateTime"),
 count(when(col("ResultType") != "0", True)).alias("FailedSignInCount")
 )
)
print(f"📊 SP credential metadata rows: {sp_cred_metadata_df.count()}")

# --- Build ServicePrincipal nodes with enrichments ---
sp_nodes_df = (
 sp_raw_df
 .select(
 col("id").alias("SpId"),
 col("appId").alias("AppId"),
 col("displayName").alias("SpDisplayName"),
 col("servicePrincipalType").alias("SpType"),
 col("accountEnabled").cast("string").alias("Enabled"),
 col("appOwnerOrganizationId").alias("OwnerTenantId"),
 col("tenantId").alias("HomeTenantId")
 )
 .distinct()
 .withColumn("IsThirdParty",
 when(
 (col("OwnerTenantId").isNotNull()) &
 (col("OwnerTenantId") != "") &
 (col("OwnerTenantId") != col("HomeTenantId")),
 lit("true")
 ).otherwise(lit("false")))
 .drop("HomeTenantId")
 # [P1.3] Join risk level
 .join(risky_sp_df, col("SpId") == col("RiskSpId"), "left")
 .drop("RiskSpId")
 .withColumn("SpRiskLevel", coalesce(col("SpRiskLevel"), lit("none")))
 .withColumn("SpRiskState", coalesce(col("SpRiskState"), lit("none")))
 .withColumn("SpRiskDetail", coalesce(col("SpRiskDetail"), lit("")))
 # [P2.2] Join credential metadata
 .join(sp_cred_metadata_df, col("SpId") == col("ServicePrincipalId"), "left")
 .drop("ServicePrincipalId")
 .withColumn("PrimaryCredentialType", coalesce(col("PrimaryCredentialType"), lit("unknown")))
 .withColumn("LastSignInDateTime", col("LastSignInDateTime").cast("string"))
 .withColumn("FailedSignInCount", coalesce(col("FailedSignInCount"), lit(0)).cast("string"))
 .withColumn("nodeType", lit("ServicePrincipal"))
)

print(f"📊 ServicePrincipal nodes: {sp_nodes_df.count()}")
print(f"📊 Third-party SPs: {sp_nodes_df.filter(col('IsThirdParty') == 'true').count()}")
print(f"📊 Risky SPs (atRisk/confirmed): {sp_nodes_df.filter(col('SpRiskState').isin('atRisk', 'confirmedCompromised')).count()}")
sp_nodes_df.show(5, truncate=False)

In [ ]:
# =============================================================================
# Cell 4: Read AuditLogs → Parse Permission Grants
# Build AppPermission nodes + User nodes
# Build HasPermission edges + GrantedBy edges
# [P0.2] Build GrantedPermissionTo edges (SP → SP)
# [P0.3] Build DirectoryRole nodes + HasDirectoryRole edges
# [P1.1] Build OwnsApp edges
# [P1.2] Build AddedSecretTo edges
# [P1.5] Enrich User nodes with risk level
# [P2.1] Build EntraGroup nodes + MemberOf edges
# [P2.3] Add temporal properties to all audit-sourced edges
# =============================================================================

# Read ALL audit logs (broader filter to support multiple edge types)
_audit_raw = sentinel_provider.read_table("AuditLogs", WORKSPACE_NAME)
_audit_raw = _audit_raw.df if hasattr(_audit_raw, 'df') else _audit_raw

audit_full_df = (
 _audit_raw
 .filter(col("TimeGenerated") >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
 .select(
 "TimeGenerated", "OperationName", "ResultType",
 "InitiatedBy", "TargetResources", "AADTenantId",
 "Category", "ActivityDisplayName"
 )
).persist()

print(f"📊 Total audit events: {audit_full_df.count()}")

# =====================================================================
# PERMISSION GRANT EVENTS (existing v1 logic)
# =====================================================================
audit_perm_df = (
 audit_full_df
 .filter(
 lower(col("OperationName")).contains("consent") |
 lower(col("OperationName")).contains("add app role assignment") |
 lower(col("OperationName")).contains("add delegated permission grant") |
 lower(col("OperationName")).contains("add oauth2permissiongrant")
 )
)

# Parse InitiatedBy
audit_parsed_df = (
 audit_perm_df
 .withColumn(
 "GrantorUserId",
 coalesce(
 get_json_object(col("InitiatedBy"), "$.user.id"),
 get_json_object(col("InitiatedBy"), "$.app.servicePrincipalId")
 )
 )
 .withColumn(
 "GrantorDisplayName",
 coalesce(
 get_json_object(col("InitiatedBy"), "$.user.userPrincipalName"),
 get_json_object(col("InitiatedBy"), "$.app.displayName")
 )
 )
 .withColumn("TargetResource0_Id",
 get_json_object(col("TargetResources"), "$[0].id"))
 .withColumn("TargetResource0_DisplayName",
 get_json_object(col("TargetResources"), "$[0].displayName"))
 .withColumn("TargetResource0_Type",
 get_json_object(col("TargetResources"), "$[0].type"))
 .withColumn("ModifiedProps",
 get_json_object(col("TargetResources"), "$[0].modifiedProperties"))
 .withColumn("TargetResource1_Id",
 get_json_object(col("TargetResources"), "$[1].id"))
 .withColumn("TargetResource1_DisplayName",
 get_json_object(col("TargetResources"), "$[1].displayName"))
)

permission_df = (
 audit_parsed_df
 .filter(col("TargetResource0_Id").isNotNull())
 .withColumn(
 "PermissionId",
 coalesce(
 concat_ws("_", col("TargetResource0_Id"), col("TargetResource0_Type")),
 col("TargetResource0_Id")
 )
 )
 .withColumn(
 "PermissionName",
 coalesce(col("TargetResource0_DisplayName"), col("OperationName"))
 )
 .withColumn(
 "TargetSpId",
 coalesce(col("TargetResource1_Id"), col("TargetResource0_Id"))
 )
).persist()

print(f"📊 Permission grant events: {permission_df.count()}")

# --- AppPermission nodes ---
permission_nodes_df = (
 permission_df
 .select(
 col("PermissionId"),
 col("PermissionName"),
 col("OperationName").alias("GrantType")
 )
 .dropDuplicates(["PermissionId"])
 .withColumn("nodeType", lit("AppPermission"))
)
print(f"📊 AppPermission nodes: {permission_nodes_df.count()}")

# --- [P1.5] Read AADRiskyUsers for user risk enrichment ---
_risky_users_raw = sentinel_provider.read_table("AADRiskyUsers", WORKSPACE_NAME)
_risky_users_raw = _risky_users_raw.df if hasattr(_risky_users_raw, 'df') else _risky_users_raw

risky_users_df = (
 _risky_users_raw
 .filter(col("TimeGenerated") >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
 .select(
 col("UserPrincipalName").alias("RiskUPN"),
 col("RiskLevel").alias("UserRiskLevel"),
 col("RiskState").alias("UserRiskState")
 )
 .dropDuplicates(["RiskUPN"])
)
print(f"📊 Risky users loaded: {risky_users_df.count()}")

# --- User nodes with risk enrichment [P1.5] ---
user_nodes_df = (
 permission_df
 .select(
 col("GrantorUserId").alias("UserId"),
 col("GrantorDisplayName").alias("UserPrincipalName")
 )
 .filter(col("UserId").isNotNull())
 .dropDuplicates(["UserId"])
 .join(risky_users_df, col("UserPrincipalName") == col("RiskUPN"), "left")
 .drop("RiskUPN")
 .withColumn("UserRiskLevel", coalesce(col("UserRiskLevel"), lit("none")))
 .withColumn("UserRiskState", coalesce(col("UserRiskState"), lit("none")))
 .withColumn("nodeType", lit("User"))
)
print(f"📊 User nodes (with risk): {user_nodes_df.count()}")

# --- HasPermission edges with temporal properties [P2.3] ---
has_permission_edges_df = (
 permission_df
 .select(
 col("TargetSpId").alias("SpId"),
 col("PermissionId"),
 col("TimeGenerated").alias("GrantedAt"),
 col("OperationName").alias("GrantType")
 )
 .filter(col("SpId").isNotNull() & col("PermissionId").isNotNull())
 .groupBy("SpId", "PermissionId")
 .agg(
 first(col("GrantType")).alias("GrantType"),
 spark_min(col("GrantedAt")).cast("string").alias("EdgeFirstSeen"),
 spark_max(col("GrantedAt")).cast("string").alias("EdgeLastSeen")
 )
 .withColumn("EdgeKey", concat_ws("_", col("SpId"), col("PermissionId")))
 .withColumn("edgeType", lit("HasPermission"))
)
print(f"📊 HasPermission edges: {has_permission_edges_df.count()}")

# --- GrantedBy edges with temporal properties [P2.3] ---
granted_by_edges_df = (
 permission_df
 .select(
 col("PermissionId"),
 col("GrantorUserId").alias("UserId"),
 col("TimeGenerated").alias("GrantedAt")
 )
 .filter(col("PermissionId").isNotNull() & col("UserId").isNotNull())
 .groupBy("PermissionId", "UserId")
 .agg(
 spark_min(col("GrantedAt")).cast("string").alias("EdgeFirstSeen"),
 spark_max(col("GrantedAt")).cast("string").alias("EdgeLastSeen")
 )
 .withColumn("EdgeKey", concat_ws("_", col("PermissionId"), col("UserId")))
 .withColumn("edgeType", lit("GrantedBy"))
)
print(f"📊 GrantedBy edges: {granted_by_edges_df.count()}")


In [ ]:
# =============================================================================
# Cell 5: [P0.2] GrantedPermissionTo Edges (SP → SP)
# [P0.3] DirectoryRole Nodes + HasDirectoryRole Edges
# [P1.1] OwnsApp Edges
# [P1.2] AddedSecretTo Edges
# [P2.1] EntraGroup Nodes + MemberOf Edges
# =============================================================================

# =====================================================================
# [P0.2] GrantedPermissionTo — SP-to-SP permission grants
# Detects: CVE-2025-55241, Midnight Blizzard step 8, Intune chain step 5
# =====================================================================
sp_grant_events_df = (
 audit_full_df
 .filter(
 lower(col("OperationName")).contains("add app role assignment") |
 lower(col("OperationName")).contains("consent") |
 lower(col("OperationName")).contains("add delegated permission grant") |
 lower(col("OperationName")).contains("add oauth2permissiongrant")
 )
 # Only events where an APP (not user) initiated the action
 .filter(get_json_object(col("InitiatedBy"), "$.app.servicePrincipalId").isNotNull())
 .withColumn("GrantorSpId",
 get_json_object(col("InitiatedBy"), "$.app.servicePrincipalId"))
 .withColumn("TargetSpId",
 coalesce(
 get_json_object(col("TargetResources"), "$[1].id"),
 get_json_object(col("TargetResources"), "$[0].id")
 ))
 .withColumn("GrantedPermission",
 coalesce(
 get_json_object(col("TargetResources"), "$[0].displayName"),
 col("OperationName")
 ))
 .filter(col("GrantorSpId").isNotNull() & col("TargetSpId").isNotNull())
)

granted_perm_to_edges_df = (
 sp_grant_events_df
 .groupBy("GrantorSpId", "TargetSpId")
 .agg(
 first(col("GrantedPermission")).alias("GrantedPermission"),
 first(col("OperationName")).alias("GrantType"),
 spark_min(col("TimeGenerated")).cast("string").alias("EdgeFirstSeen"),
 spark_max(col("TimeGenerated")).cast("string").alias("EdgeLastSeen"),
 count("*").alias("GrantCount")
 )
 .withColumn("GrantCount", col("GrantCount").cast("string"))
 .withColumn("EdgeKey", concat_ws("_", col("GrantorSpId"), col("TargetSpId")))
 .withColumn("edgeType", lit("GrantedPermissionTo"))
)
print(f"📊 [P0.2] GrantedPermissionTo edges (SP→SP): {granted_perm_to_edges_df.count()}")

# =====================================================================
# [P0.3] DirectoryRole Nodes + HasDirectoryRole Edges
# Detects: Tier Zero SPs, PIM bypass, shortest-path-to-GlobalAdmin
# =====================================================================
role_events_df = (
 audit_full_df
 .filter(
 lower(col("OperationName")).contains("add member to role") |
 lower(col("OperationName")).contains("add eligible member to role")
 )
 .withColumn("RoleName",
 get_json_object(col("TargetResources"), "$[0].displayName"))
 .withColumn("RoleId",
 get_json_object(col("TargetResources"), "$[0].id"))
 .withColumn("MemberId",
 coalesce(
 get_json_object(col("TargetResources"), "$[1].id"),
 get_json_object(col("TargetResources"), "$[0].id")
 ))
 .withColumn("AssignmentType",
 when(lower(col("OperationName")).contains("eligible"), lit("PIM-Eligible"))
 .otherwise(lit("Permanent")))
 .filter(col("RoleId").isNotNull() & col("MemberId").isNotNull())
)

# DirectoryRole nodes
directory_role_nodes_df = (
 role_events_df
 .select(col("RoleId"), col("RoleName"))
 .dropDuplicates(["RoleId"])
 .withColumn("IsTierZero",
 when(lower(col("RoleName")).isin(TIER_ZERO_ROLES), lit("true"))
 .otherwise(lit("false")))
 .withColumn("nodeType", lit("DirectoryRole"))
)
print(f"📊 [P0.3] DirectoryRole nodes: {directory_role_nodes_df.count()}")
print(f"📊 Tier Zero roles: {directory_role_nodes_df.filter(col('IsTierZero') == 'true').count()}")

# HasDirectoryRole edges (SP/User → DirectoryRole)
has_dir_role_edges_df = (
 role_events_df
 .groupBy("MemberId", "RoleId")
 .agg(
 first(col("AssignmentType")).alias("AssignmentType"),
 first(col("RoleName")).alias("RoleName"),
 spark_min(col("TimeGenerated")).cast("string").alias("EdgeFirstSeen"),
 spark_max(col("TimeGenerated")).cast("string").alias("EdgeLastSeen")
 )
 .withColumn("EdgeKey", concat_ws("_", col("MemberId"), col("RoleId")))
 .withColumn("edgeType", lit("HasDirectoryRole"))
)
print(f"📊 [P0.3] HasDirectoryRole edges: {has_dir_role_edges_df.count()}")

# =====================================================================
# [P1.1] OwnsApp Edges (User/SP → SP)
# Detects: Application Admin → Global Admin privesc via owner path
# =====================================================================
owns_events_df = (
 audit_full_df
 .filter(
 lower(col("OperationName")).contains("add owner to application") |
 lower(col("OperationName")).contains("add owner to service principal")
 )
 .withColumn("OwnerId",
 coalesce(
 get_json_object(col("InitiatedBy"), "$.user.id"),
 get_json_object(col("InitiatedBy"), "$.app.servicePrincipalId")
 ))
 .withColumn("OwnedSpId",
 coalesce(
 get_json_object(col("TargetResources"), "$[0].id"),
 get_json_object(col("TargetResources"), "$[1].id")
 ))
 .filter(col("OwnerId").isNotNull() & col("OwnedSpId").isNotNull())
)

owns_app_edges_df = (
 owns_events_df
 .groupBy("OwnerId", "OwnedSpId")
 .agg(
 spark_min(col("TimeGenerated")).cast("string").alias("EdgeFirstSeen"),
 spark_max(col("TimeGenerated")).cast("string").alias("EdgeLastSeen")
 )
 .withColumn("EdgeKey", concat_ws("_", col("OwnerId"), col("OwnedSpId")))
 .withColumn("edgeType", lit("OwnsApp"))
)
print(f"📊 [P1.1] OwnsApp edges: {owns_app_edges_df.count()}")

# =====================================================================
# [P1.2] AddedSecretTo Edges (User/SP → SP)
# Detects: Credential addition as backdoor precondition
# =====================================================================
secret_events_df = (
 audit_full_df
 .filter(
 lower(col("OperationName")).contains("certificates and secrets management") |
 lower(col("OperationName")).contains("add service principal credentials") |
 lower(col("OperationName")).contains("update application")
 )
 .withColumn("ActorId",
 coalesce(
 get_json_object(col("InitiatedBy"), "$.user.id"),
 get_json_object(col("InitiatedBy"), "$.app.servicePrincipalId")
 ))
 .withColumn("TargetAppId",
 get_json_object(col("TargetResources"), "$[0].id"))
 .filter(col("ActorId").isNotNull() & col("TargetAppId").isNotNull())
)

added_secret_edges_df = (
 secret_events_df
 .groupBy("ActorId", "TargetAppId")
 .agg(
 spark_min(col("TimeGenerated")).cast("string").alias("EdgeFirstSeen"),
 spark_max(col("TimeGenerated")).cast("string").alias("EdgeLastSeen"),
 count("*").alias("SecretAddCount")
 )
 .withColumn("SecretAddCount", col("SecretAddCount").cast("string"))
 .withColumn("EdgeKey", concat_ws("_", col("ActorId"), col("TargetAppId")))
 .withColumn("edgeType", lit("AddedSecretTo"))
)
print(f"📊 [P1.2] AddedSecretTo edges: {added_secret_edges_df.count()}")

# =====================================================================
# [P2.1] EntraGroup Nodes + MemberOf Edges (SP → Group)
# Detects: Hidden privilege via group membership inheritance
# =====================================================================
group_events_df = (
 audit_full_df
 .filter(
 lower(col("OperationName")).contains("add member to group")
 )
 .withColumn("GroupId",
 get_json_object(col("TargetResources"), "$[0].id"))
 .withColumn("GroupName",
 get_json_object(col("TargetResources"), "$[0].displayName"))
 .withColumn("MemberId",
 coalesce(
 get_json_object(col("TargetResources"), "$[1].id"),
 get_json_object(col("TargetResources"), "$[0].id")
 ))
 .filter(col("GroupId").isNotNull() & col("MemberId").isNotNull())
)

# EntraGroup nodes
entra_group_nodes_df = (
 group_events_df
 .select(col("GroupId"), col("GroupName"))
 .dropDuplicates(["GroupId"])
 .withColumn("nodeType", lit("EntraGroup"))
)
print(f"📊 [P2.1] EntraGroup nodes: {entra_group_nodes_df.count()}")

# MemberOf edges
member_of_edges_df = (
 group_events_df
 .groupBy("MemberId", "GroupId")
 .agg(
 spark_min(col("TimeGenerated")).cast("string").alias("EdgeFirstSeen"),
 spark_max(col("TimeGenerated")).cast("string").alias("EdgeLastSeen")
 )
 .withColumn("EdgeKey", concat_ws("_", col("MemberId"), col("GroupId")))
 .withColumn("edgeType", lit("MemberOf"))
)
print(f"📊 [P2.1] MemberOf edges: {member_of_edges_df.count()}")


In [ ]:
# =============================================================================
# Cell 6: [P1.4] SP Risk Events — RiskEvent Nodes + HasRiskEvent Edges
# =============================================================================

_risk_events_raw = sentinel_provider.read_table("AADServicePrincipalRiskEvents", WORKSPACE_NAME)
_risk_events_raw = _risk_events_raw.df if hasattr(_risk_events_raw, 'df') else _risk_events_raw

risk_events_df = (
 _risk_events_raw
 .filter(col("TimeGenerated") >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
 .select(
 col("Id").alias("RiskEventId"),
 col("ServicePrincipalId").alias("RiskSpId"),
 col("RiskEventType"),
 col("RiskLevel").alias("EventRiskLevel"),
 col("RiskState").alias("EventRiskState"),
 col("DetectionTimingType"),
 col("IpAddress").alias("RiskSourceIP"),
 col("DetectedDateTime").cast("string").alias("DetectedAt")
 )
 .filter(col("RiskEventId").isNotNull())
 .dropDuplicates(["RiskEventId"])
).persist()

print(f"📊 SP Risk Events loaded: {risk_events_df.count()}")

# RiskEvent nodes
risk_event_nodes_df = (
 risk_events_df
 .select(
 col("RiskEventId"),
 col("RiskEventType"),
 col("EventRiskLevel"),
 col("EventRiskState"),
 col("DetectionTimingType"),
 col("RiskSourceIP"),
 col("DetectedAt")
 )
 .withColumn("nodeType", lit("RiskEvent"))
)
print(f"📊 [P1.4] RiskEvent nodes: {risk_event_nodes_df.count()}")

# HasRiskEvent edges (SP → RiskEvent)
has_risk_event_edges_df = (
 risk_events_df
 .select(
 col("RiskSpId").alias("SpId"),
 col("RiskEventId"),
 col("DetectedAt").alias("EdgeFirstSeen")
 )
 .filter(col("SpId").isNotNull())
 .withColumn("EdgeLastSeen", col("EdgeFirstSeen"))
 .withColumn("EdgeKey", concat_ws("_", col("SpId"), col("RiskEventId")))
 .withColumn("edgeType", lit("HasRiskEvent"))
)
print(f"📊 [P1.4] HasRiskEvent edges: {has_risk_event_edges_df.count()}")


In [ ]:
# =============================================================================
# Cell 7: Read MicrosoftGraphActivityLogs
# [P0.1] FIXED CalledAPI regex + ServicePrincipalId direct usage
# [P2.4] CalledSensitiveAPI filtered edge
# [P2.3] Temporal properties on edges
# =============================================================================

_graph_api_raw = sentinel_provider.read_table("MicrosoftGraphActivityLogs", WORKSPACE_NAME)
_graph_api_raw = _graph_api_raw.df if hasattr(_graph_api_raw, 'df') else _graph_api_raw

graph_api_raw_df = (
 _graph_api_raw
 .filter(col("TimeGenerated") >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
 .select(
 "TimeGenerated", "AppId", "ServicePrincipalId",
 "RequestUri", "RequestMethod", "ResponseStatusCode",
 "IPAddress", "Roles", "Scopes"
 )
).persist()

print(f"📊 Graph API activity logs: {graph_api_raw_df.count()}")

# [P0.1] FIXED: Handle both full URLs (https://graph.microsoft.com/v1.0/...)
# and relative paths (/v1.0/...). Previous regex anchored at ^/ dropped all full URLs.
graph_api_parsed_df = (
 graph_api_raw_df
 # Remove query parameters first
 .withColumn("UriNoQuery", split(col("RequestUri"), "\\?").getItem(0))
 # [P0.1 FIX] Handle both full URLs and relative paths
 .withColumn("CleanUri",
 regexp_extract(col("UriNoQuery"),
 r"(?:https?://[^/]+)?(/(?:v1\.0|beta)/.+?)$", 1))
 # Extract top-level endpoint pattern
 .withColumn("Endpoint",
 regexp_extract(col("CleanUri"),
 r"^(/[^/]+(?:/[^/]+)?)", 1))
 .filter(col("Endpoint") != "")
 .filter(col("Endpoint").isNotNull())
)

# [P0.1 FIX] Use ServicePrincipalId directly instead of AppId join
# The AppId→SP join failed when AppId didn't match EntraServicePrincipals
sp_lookup_df = sp_raw_df.select(col("appId").alias("lookupAppId"), col("id").alias("SpIdResolved"))

graph_api_agg_df = (
 graph_api_parsed_df
 .withColumn("CallDate", to_date(col("TimeGenerated")))
 # [P0.1 FIX] Prefer ServicePrincipalId, fall back to AppId lookup
 .join(sp_lookup_df, col("AppId") == col("lookupAppId"), "left")
 .withColumn("ResolvedSpId",
 coalesce(col("ServicePrincipalId"), col("SpIdResolved"), col("AppId")))
 .groupBy(
 col("ResolvedSpId").alias("SpId"),
 col("Endpoint"),
 col("CallDate")
 )
 .agg(
 count("*").alias("CallCount"),
 first(col("RequestMethod")).alias("PrimaryMethod")
 )
).persist()

print(f"📊 Aggregated API activity: {graph_api_agg_df.count()} rows")
graph_api_agg_df.show(10, truncate=False)

# --- Resource nodes ---
resource_nodes_df = (
 graph_api_agg_df
 .select(col("Endpoint"))
 .distinct()
 .withColumn("nodeType", lit("Resource"))
)
print(f"📊 Resource nodes (API endpoints): {resource_nodes_df.count()}")

# --- CalledAPI edges with temporal properties [P2.3] ---
called_api_edges_df = (
 graph_api_agg_df
 .groupBy("SpId", "Endpoint")
 .agg(
 spark_sum(col("CallCount")).alias("TotalCallCount"),
 first(col("PrimaryMethod")).alias("PrimaryMethod"),
 spark_min(col("CallDate")).cast("string").alias("EdgeFirstSeen"),
 spark_max(col("CallDate")).cast("string").alias("EdgeLastSeen")
 )
 .withColumn("TotalCallCount", col("TotalCallCount").cast("string"))
 .filter(col("SpId").isNotNull() & col("Endpoint").isNotNull())
 .withColumn("EdgeKey", concat_ws("_", col("SpId"), col("Endpoint")))
 .withColumn("edgeType", lit("CalledAPI"))
)
print(f"📊 CalledAPI edges: {called_api_edges_df.count()}")

# =====================================================================
# [P2.4] CalledSensitiveAPI — Filtered high-fidelity subset
# =====================================================================
sensitive_pattern = "|".join(SENSITIVE_API_PATTERNS)

called_sensitive_api_edges_df = (
 graph_api_parsed_df
 .filter(col("CleanUri").rlike("(?i)(" + "|".join(
 [p.replace("*", "[^/]+").replace("/", "\\/") for p in SENSITIVE_API_PATTERNS]
 ) + ")"))
 .join(sp_lookup_df, col("AppId") == col("lookupAppId"), "left")
 .withColumn("SpId",
 coalesce(col("ServicePrincipalId"), col("SpIdResolved"), col("AppId")))
 .withColumn("SensitiveEndpoint",
 regexp_extract(col("CleanUri"), r"^(/[^/]+(?:/[^/]+(?:/[^/]+)?)?)", 1))
 .groupBy("SpId", "SensitiveEndpoint")
 .agg(
 count("*").alias("SensitiveCallCount"),
 first(col("RequestMethod")).alias("PrimaryMethod"),
 spark_min(col("TimeGenerated")).cast("string").alias("EdgeFirstSeen"),
 spark_max(col("TimeGenerated")).cast("string").alias("EdgeLastSeen")
 )
 .withColumn("SensitiveCallCount", col("SensitiveCallCount").cast("string"))
 .filter(col("SpId").isNotNull() & col("SensitiveEndpoint").isNotNull())
 .withColumn("EdgeKey", concat_ws("_", col("SpId"), col("SensitiveEndpoint")))
 .withColumn("edgeType", lit("CalledSensitiveAPI"))
)
print(f"📊 [P2.4] CalledSensitiveAPI edges: {called_sensitive_api_edges_df.count()}")


In [ ]:
# =============================================================================
# Cell 8: Read AADServicePrincipalSignInLogs
# Build SourceIP nodes + AuthenticatedAs edges
# [P2.3] Add temporal properties
# =============================================================================

_sp_signin_raw = sentinel_provider.read_table("AADServicePrincipalSignInLogs", WORKSPACE_NAME)
_sp_signin_raw = _sp_signin_raw.df if hasattr(_sp_signin_raw, 'df') else _sp_signin_raw

sp_signin_raw_df = (
 _sp_signin_raw
 .filter(col("TimeGenerated") >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
 .select(
 "TimeGenerated", "ServicePrincipalId", "ServicePrincipalName",
 "IPAddress", "ResourceDisplayName", "ResultType",
 "ClientCredentialType", "Location"
 )
).persist()

print(f"📊 SP sign-in logs: {sp_signin_raw_df.count()}")

# --- SourceIP nodes ---
source_ip_nodes_df = (
 sp_signin_raw_df
 .filter(col("IPAddress").isNotNull())
 .select(col("IPAddress"))
 .distinct()
 .withColumn("nodeType", lit("SourceIP"))
)
print(f"📊 SourceIP nodes: {source_ip_nodes_df.count()}")

# --- AuthenticatedAs edges with temporal properties [P2.3] ---
authenticated_as_edges_df = (
 sp_signin_raw_df
 .filter(col("IPAddress").isNotNull())
 .groupBy("ServicePrincipalId", "IPAddress")
 .agg(
 count("*").alias("SignInCount"),
 first(col("ResourceDisplayName")).alias("ResourceDisplayName"),
 first(col("ClientCredentialType")).alias("CredentialType"),
 first(col("Location")).alias("Location"),
 spark_min(col("TimeGenerated")).cast("string").alias("EdgeFirstSeen"),
 spark_max(col("TimeGenerated")).cast("string").alias("EdgeLastSeen")
 )
 .withColumn("SignInCount", col("SignInCount").cast("string"))
 .withColumn("SpId", col("ServicePrincipalId"))
 .withColumn("EdgeKey", concat_ws("_", col("SpId"), col("IPAddress")))
 .withColumn("edgeType", lit("AuthenticatedAs"))
)
print(f"📊 AuthenticatedAs edges: {authenticated_as_edges_df.count()}")


## Graph Assembly

Assemble the graph using the GraphSpecBuilder fluent API.


In [ ]:
# =============================================================================
# Cell 9: Assemble Graph with GraphSpecBuilder
# =============================================================================

oauth_sp_graph = (GraphSpecBuilder.start()

 # ===================== NODES (9 types) =====================

 .add_node("ServicePrincipal")
 .from_dataframe(sp_nodes_df)
 .with_columns(
 "SpId", "AppId", "SpDisplayName", "SpType", "Enabled", "OwnerTenantId",
 "IsThirdParty",
 "SpRiskLevel", "SpRiskState", "SpRiskDetail",
 "PrimaryCredentialType", "LastSignInDateTime",
 "FailedSignInCount",
 "nodeType",
 key="SpId", display="SpDisplayName"
 )

 .add_node("AppPermission")
 .from_dataframe(permission_nodes_df)
 .with_columns(
 "PermissionId", "PermissionName", "GrantType", "nodeType",
 key="PermissionId", display="PermissionName"
 )

 .add_node("User")
 .from_dataframe(user_nodes_df)
 .with_columns(
 "UserId", "UserPrincipalName",
 "UserRiskLevel", "UserRiskState",
 "nodeType",
 key="UserId", display="UserPrincipalName"
 )

 .add_node("Resource")
 .from_dataframe(resource_nodes_df)
 .with_columns(
 "Endpoint", "nodeType",
 key="Endpoint", display="Endpoint"
 )

 .add_node("SourceIP")
 .from_dataframe(source_ip_nodes_df)
 .with_columns(
 "IPAddress", "nodeType",
 key="IPAddress", display="IPAddress"
 )

 .add_node("DirectoryRole")
 .from_dataframe(directory_role_nodes_df)
 .with_columns(
 "RoleId", "RoleName", "IsTierZero", "nodeType",
 key="RoleId", display="RoleName"
 )

 .add_node("RiskEvent")
 .from_dataframe(risk_event_nodes_df)
 .with_columns(
 "RiskEventId", "RiskEventType", "EventRiskLevel", "EventRiskState",
 "DetectionTimingType", "RiskSourceIP", "DetectedAt", "nodeType",
 key="RiskEventId", display="RiskEventType"
 )

 .add_node("EntraGroup")
 .from_dataframe(entra_group_nodes_df)
 .with_columns(
 "GroupId", "GroupName", "nodeType",
 key="GroupId", display="GroupName"
 )

 # ===================== EDGES (11 types) =====================

 .add_edge("HasPermission")
 .from_dataframe(has_permission_edges_df)
 .source(id_column="SpId", node_type="ServicePrincipal")
 .target(id_column="PermissionId", node_type="AppPermission")
 .with_columns(
 "EdgeKey", "GrantType", "EdgeFirstSeen", "EdgeLastSeen", "edgeType",
 key="EdgeKey", display="edgeType"
 )

 .add_edge("GrantedBy")
 .from_dataframe(granted_by_edges_df)
 .source(id_column="UserId", node_type="User")
 .target(id_column="PermissionId", node_type="AppPermission")
 .with_columns(
 "EdgeKey", "EdgeFirstSeen", "EdgeLastSeen", "edgeType",
 key="EdgeKey", display="edgeType"
 )

 .add_edge("CalledAPI")
 .from_dataframe(called_api_edges_df)
 .source(id_column="SpId", node_type="ServicePrincipal")
 .target(id_column="Endpoint", node_type="Resource")
 .with_columns(
 "EdgeKey", "TotalCallCount", "PrimaryMethod",
 "EdgeFirstSeen", "EdgeLastSeen", "edgeType",
 key="EdgeKey", display="edgeType"
 )

 .add_edge("AuthenticatedAs")
 .from_dataframe(authenticated_as_edges_df)
 .source(id_column="SpId", node_type="ServicePrincipal")
 .target(id_column="IPAddress", node_type="SourceIP")
 .with_columns(
 "EdgeKey", "SignInCount", "ResourceDisplayName", "CredentialType",
 "EdgeFirstSeen", "EdgeLastSeen", "edgeType",
 key="EdgeKey", display="edgeType"
 )

 .add_edge("GrantedPermissionTo")
 .from_dataframe(granted_perm_to_edges_df)
 .source(id_column="GrantorSpId", node_type="ServicePrincipal")
 .target(id_column="TargetSpId", node_type="ServicePrincipal")
 .with_columns(
 "EdgeKey", "GrantedPermission", "GrantType", "GrantCount",
 "EdgeFirstSeen", "EdgeLastSeen", "edgeType",
 key="EdgeKey", display="edgeType"
 )

 .add_edge("HasDirectoryRole")
 .from_dataframe(has_dir_role_edges_df)
 .source(id_column="MemberId", node_type="ServicePrincipal")
 .target(id_column="RoleId", node_type="DirectoryRole")
 .with_columns(
 "EdgeKey", "AssignmentType", "RoleName",
 "EdgeFirstSeen", "EdgeLastSeen", "edgeType",
 key="EdgeKey", display="edgeType"
 )

 .add_edge("OwnsApp")
 .from_dataframe(owns_app_edges_df)
 .source(id_column="OwnerId", node_type="User")
 .target(id_column="OwnedSpId", node_type="ServicePrincipal")
 .with_columns(
 "EdgeKey", "EdgeFirstSeen", "EdgeLastSeen", "edgeType",
 key="EdgeKey", display="edgeType"
 )

 .add_edge("AddedSecretTo")
 .from_dataframe(added_secret_edges_df)
 .source(id_column="ActorId", node_type="User")
 .target(id_column="TargetAppId", node_type="ServicePrincipal")
 .with_columns(
 "EdgeKey", "SecretAddCount", "EdgeFirstSeen", "EdgeLastSeen", "edgeType",
 key="EdgeKey", display="edgeType"
 )

 .add_edge("HasRiskEvent")
 .from_dataframe(has_risk_event_edges_df)
 .source(id_column="SpId", node_type="ServicePrincipal")
 .target(id_column="RiskEventId", node_type="RiskEvent")
 .with_columns(
 "EdgeKey", "EdgeFirstSeen", "EdgeLastSeen", "edgeType",
 key="EdgeKey", display="edgeType"
 )

 .add_edge("MemberOf")
 .from_dataframe(member_of_edges_df)
 .source(id_column="MemberId", node_type="ServicePrincipal")
 .target(id_column="GroupId", node_type="EntraGroup")
 .with_columns(
 "EdgeKey", "EdgeFirstSeen", "EdgeLastSeen", "edgeType",
 key="EdgeKey", display="edgeType"
 )

 .add_edge("CalledSensitiveAPI")
 .from_dataframe(called_sensitive_api_edges_df)
 .source(id_column="SpId", node_type="ServicePrincipal")
 .target(id_column="SensitiveEndpoint", node_type="Resource")
 .with_columns(
 "EdgeKey", "SensitiveCallCount", "PrimaryMethod",
 "EdgeFirstSeen", "EdgeLastSeen", "edgeType",
 key="EdgeKey", display="edgeType"
 )

).done()



In [ ]:
# Check the schema of the graph spec to ensure it is correct
oauth_sp_graph.show_schema()


## Build & Publish

Build the graph from the spec. This validates the schema, prepares the data, and publishes the graph for querying.


In [ ]:
# Build the graph from the spec - this will validate the spec and prepare it for querying
# Alternative: use Graph.prepare() to prepare the graph nodes and edges in the lake
# and then use Graph.publish() to create the graph. You would typically call prepare()
# and publish() separately to understand the cost of Graph API calls triggered by Graph.publish()
# see https://learn.microsoft.com/azure/sentinel/billing?tabs=simplified%2Ccommitment-tiers
oauth_graph = Graph.build(oauth_sp_graph)
